# NEXUS Research · APTOS Diabetic Retinopathy

**Offline evidence notebook | open-source research | not a medical device**

This notebook reads versioned model evaluation reports already committed to the repository. It **does not** perform new training, use Kaggle competition test labels, require BigQuery, or measure an independent validation result.

**Run in this repository root**, after `pip install -e .` (for Kaggle, add or clone this repo to the notebook working directory first). Data in `reports/` is sufficient; no fundus images or GPU are needed.

Analyses: provenance fingerprint, CV mean/ensemble differences, external-domain sensitivity and Wilson 95% descriptive intervals, research limitations.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aptos.reporting.research_evidence import build_evidence

cwd = Path.cwd()
root = next((p for p in (cwd, *cwd.parents) if (p / "reports" / "runs.csv").exists()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the APTOS GitHub repository, including reports/.")

evidence = build_evidence(root / "reports")
print(f"Audited {len(evidence['cross_validation'])} CV sweeps and {len(evidence['external'])} external cohorts.")
print("Input SHA-256 hashes:")
for name, digest in evidence["inputs_sha256"].items():
    print(f"  {name}: {digest}")

## 1. Recorded five-fold experiments

Fold-mean test QWK is **not** an independent performance estimate of the five-fold ensemble. The latter is measured on the same held-out APTOS test set; plot them side-by-side only to describe the recorded runs. APTOS test results are not a hyperparameter-selection source.

In [ ]:
cv = pd.DataFrame(evidence["cross_validation"]).sort_values(["variant", "sweep"])
display(cv[["variant", "sweep", "folds", "fold_test_qwk_mean", "fold_test_qwk_sd", "ensemble_test_qwk"]])

labels = [f"{v} / {s[:6]}" for v, s in zip(cv["variant"], cv["sweep"], strict=True)]
positions = np.arange(len(cv))
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(positions - .18, cv["fold_test_qwk_mean"], width=.36, color="#4EA8FF", label="Fold mean")
ax.bar(positions + .18, cv["ensemble_test_qwk"], width=.36, color="#8F9BFF", label="Ensemble")
ax.set_xticks(positions, labels, rotation=20, ha="right")
ax.set_ylim(0, 1.0)
ax.set_ylabel("Quadratic weighted kappa (QWK)")
ax.set_title("APTOS held-out test · descriptive model comparisons")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 2. External cohorts: fixed APTOS thresholds

IDRiD and Messidor-2 have different acquisition/preprocessing and grading settings. We plot observed sensitivity **with unchanged APTOS thresholds**. Wilson intervals assume independent eyes and are **not** patient-cluster-adjusted. They are descriptive, not proof of clinical transportability.

In [ ]:
external = pd.DataFrame(evidence["external"])
display(external[["dataset", "n", "referable", "missed_referable", "sensitivity", "specificity", "qwk"]])

x = np.arange(len(external))
sens = external["sensitivity"].to_numpy()
bounds = external["sensitivity_wilson95"].tolist()
lower = np.array([a for a, b in bounds])
upper = np.array([b for a, b in bounds])
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(x, sens, yerr=[sens - lower, upper - sens], fmt="o", markersize=10,
            capsize=6, color="#4EA8FF", ecolor="#8F9BFF", linewidth=2)
ax.set_xticks(x, external["dataset"])
ax.set_ylim(0, 1)
ax.set_ylabel("Referable sensitivity (grade ≥ 2)")
ax.set_title("External sensitivity · original thresholds · descriptive Wilson intervals")
ax.grid(axis="y", alpha=.15)
for i, row in enumerate(external.itertuples()):
    ax.annotate(f"{row.missed_referable} missed / {row.referable} referable",
                (i, row.sensitivity), xytext=(0, 18), textcoords="offset points",
                ha="center", fontsize=9)
fig.tight_layout()
plt.show()

## 3. Negative results and evidence boundaries

- **Metadata shortcut:** A RandomForest using only file metadata obtained QWK ≈ 0.652 in the project's prior analysis. A classifier can exploit acquisition signatures.
- **Duplicate-label disagreement:** 43 of 148 exact-image pairs carry conflicting labels; the 84% "ceiling" approximation is not an identified clinical or population-wide upper bound.
- **External failure:** Messidor-2 referral sensitivity was 150/457 ≈ 32.8%, markedly worse than IDRiD. Cross-cohort differences do not uniquely establish a causal mechanism.
- **Unexecuted work:** This notebook does not reproduce GPU model training, download images or measure any new model performance.
- **Intended use:** Research and methods demonstration only. No diagnostic or referral decisions.

In [ ]:
assert evidence["research_only"] is True
assert all(row["thresholds_refitted"] is False for row in evidence["external"])
print(f"Observed IDRiD–Messidor-2 sensitivity gap: {evidence['external_sensitivity_gap_percentage_points']} percentage points.")
print("All source data fingerprints and aggregate consistency checks passed.")